# Three-field multiscale Darcy: MH²M

Two independent interface spaces implement equations (25)–(29) of [de Barros, Madureira and Valentin, arXiv:2404.16978v3](https://arxiv.org/abs/2404.16978v3). The global unknown is the continuous pressure trace Γ. The local conormal Λ uses λ=A∇p·n, the negative of the physical outward Darcy flux.

In [ ]:
from pathlib import Path
import json
import numpy as np
from IPython.display import Image, display
from threadpoolctl import threadpool_limits
from pymhm.mesh import TriangleMesh, SkeletonSpace, FaceSpace
from pymhm.mh2m import PressureTraceSpace, solve_mh2m

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent

## Small analytical verification

A nonharmonic quadratic with nonzero Dirichlet data checks the complete source reconstruction. The independent campaign below uses the quartic pressure from Section 8.1.

In [ ]:
def exact(x):
    """Quadratic pressure with nonzero boundary values."""
    return x[:, 0]**2 + x[:, 0]*x[:, 1] + 2*x[:, 1]**2

mesh = TriangleMesh.unit_square(2)
with threadpool_limits(1):
    solution = solve_mh2m(
        mesh, source=-6, dirichlet=exact, degree=2, local_refinement=2,
        pressure_trace=PressureTraceSpace.uniform(mesh, 2),
        flux_space=SkeletonSpace(mesh, tuple(FaceSpace.uniform(1) for _ in mesh.faces)),
    )
assert solution.l2_error(exact) < 1e-11
assert np.max(abs(solution.conservation_residuals())) < 1e-11
{"pressure_error_l2": solution.l2_error(exact), "global_residual": solution.residual}

## Smooth published data and independent space refinements

The full study contains three polynomial families and five macro resolutions. The oscillatory coefficient uses equation (61), with explicitly selected γ=1 and the smooth source. The article does not identify these historical data completely; this is a declared-data experiment, not equality with its plotted fields. The numerical P1 reference has its own refinement study.

In [ ]:
record = json.loads((ROOT / "examples/results/mh2m/comparison.json").read_text())
pressure = json.loads((ROOT / "examples/results/mh2m/pressure-enrichment.json").read_text())
assert record["source_changed_during_run"] is False
assert pressure["source_changed_during_run"] is False
[(row["flux_segments"], row["free_trace_dofs"], row["flux_relative_difference"])
 for row in record["enrichment"]]

Conormal enrichment leaves nine free global unknowns and does not reduce the error of the unresolved fixed pressure trace. Refining Γ separately reduces the flux difference from 38.05% to 4.334%; the last classical-reference refinement changes the flux by 4.170%. A small algebraic residual does not remove this approximation/reference uncertainty.

In [ ]:
for name in ["convergence", "smooth-fields", "enrichment", "pressure-trace-enrichment",
             "oscillatory-pressure", "oscillatory-flux"]:
    display(Image(filename=str(ROOT / "docs/figures/mh2m" / f"{name}.png")))

Reacquire with `pixi run -e notebooks python -m examples.mh2m_campaign` and `pixi run -e notebooks python -m examples.mh2m_trace_campaign`. Render archived data with `pixi run -e notebooks python -m examples.plot_mh2m`. The source notebook executes only the small quadratic check; numerical campaigns remain separate from light CI.